# BPE Tokenizer

This program is inspired by https://github.com/karpathy/minbpe.  
The steps can be found in `exercise.md`.  

. . .  

Independently coded by Justin Mo without assistance as additional BPE practice.  
**NOTE:** all of my pseudocode and thoughts will be done in *italics*.

---
**Step 1**  

Write the BasicTokenizer class, with the following three core functions:

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`
---


*The first step is to iterate through the entire sequence (which is the list `ids`), then pair up each adjacent token. Then, we add it to a dictionary with the key being the pair (a tuple) and the value being the frequency of the pair. When we being merging, we want to start with the most frequent pairs.*

In [ ]:
def get_stats(ids): # stats represents the statistics about the data, specifically the frequency count of pairs
  stats = {}
  for i in range(len(ids) - 1):
    pair = (ids[i], ids[i + 1])
    stats[pair] = stats.get(pair, 0) + 1  # stats.get(pair, 0) gets the key value. If introduced for the first time, it will set the key to 0. Otherwise, value++

  return stats

*Now that we have the most frequent pairs, we want to actually merge them. So, we create a `merge()` function with three parameters: `ids` (tokenized list), `pair` (the pair we want to merge), `new_id` (what we want to replace the pair with).*  
*First, we want to go through the list. If `ids[i]` and `ids[i + 1]` match up perfectly with the pair (make sure that the ordering is correct), then we want to just add `new_id` to our new list. If not, then add `ids[i]`. We ignore `ids[i + 1]` for now because we repeat the iteration again, when `i = i + 1`. We must note that the last element may sometimes be overlooked if it is not the pair we want to replace. In this case, we just add it to the very end of our list.*

In [ ]:
# take a list of 'ids', and replace every instance of 'pair' with 'new_id'
def merge(ids, pair, new_id):
  merged_stats = []

  i = 0
  while i < len(ids):

    # edge case -- prevents ids[i + 1] from going out of bounds
    if i == len(ids) - 1:
      merged_stats.append(ids[i])
      break

    if ids[i] == pair[0] and ids[i + 1] == pair[1]:
      merged_stats.append(new_id)
      i += 2
    else:
      merged_stats.append(ids[i])
      i += 1

  return merged_stats

*Refer to the snippet below for additional information.*

In [ ]:
print("hello".encode("utf-8"))        # becomes a byte object
print(list("hello".encode("utf-8")))  # wrapping it in a list turns them into a list of integers

b'hello'
[104, 101, 108, 108, 111]


*Now that we have our helper functions completed, we can tackle the training loop. The first step is to convert the text into tokens. We can use the code `text.encode("utf-8")`. However, this gives us bytes. We resolve this by wrapping it with `list()`, turning them into integer representations.*

*We need to initialize two lists: `merges` and `vocab`, where the first tracks all the merges and the latter tracks what the merge is. For `vocab`, we already know what the first 256 vocabulary tokens are: the byte representation from 0 to 255. So, from `0 <= i < 256`, we set `self.vocab[i] = bytes([i])`.*

*In our training loop, we want to run it `vocab_size - 256` times to get a total of `vocab_size` vocabulary (our first 256 was already previously defined). Then we just apply our helper functions. Lastly, we want to actually save it to our `merges` and `vocab` dictionary.*

In [ ]:
def train(self, text, vocab_size, verbose = False):
  ids = list(text.encode("utf-8"))

  self.merges = {}
  self.vocab = {}
  for i in range(256):
    self.vocab[i] = bytes([i])

  for i in range(vocab_size - 256):
    stats = get_stats(ids)
    pair = max(stats, key = stats.get)  # returns the key pair but ranks them by value (highest value = what we want to merge)
    ids = merge(ids, pair, 256 + i)

    self.merges[pair] = 256 + i # maps the pair to the token value
    self.vocab[256 + i] = self.vocab[pair[0]] + self.vocab[pair[1]] # actually adds the two tokens together


*Let's decode. Right now, I want to convert the entirety of `ids` back into its string format. To do so, we want to join together ALL of the tokens and then decode it into string format via `utf-8`. To do so, we use `self.vocab[id] for id in ids`. This will get all token values from `ids` and then convert them via `self.vocab`, which houses the token translations. By iterating through every value in `ids`, we can join them to recreate the full text. Note that we use `b""` as the seperator for `.join()`, meaning all the bytes get concatenated with nothing in between.*

In [ ]:
def decode(self, ids):
  byte_seq = b"".join(self.vocab[id] for id in ids) # takes the 'b' prefix out of all the bytes and joins them together to get our original string
  return byte_seq.decode("utf-8")

*Lastly, we get to work on the encode function. Remember that encoding is just the reverse of decoding; given a string, I want to return a integer list we name `ids`. So, as always, we always create the encoded list via `ids = list(text.encode("utf-8"))`. Then, we want to keep merging until we can no longer merge (in essence, performing every possible merge).*

*In merging, it is completely possible that a merged token gets used again for another merging sequence. Therefore, we cannot ignore the order of how these tokens are merged. As such, we need to start by merging the pair with the lowest merge index (in effect, our first merge). We set `self.merges.get(p, float("inf"))` to set all unlearned merges to infinity, so that the minimum function will never try to merge them.*

*Finally, we save it to `ids` and return it.*

In [ ]:
def encode(self, text):
  ids = list(text.encode("utf-8"))

  while(True):
    stats = get_stats(ids)
    if not any(p in self.merges for p in stats):  # if none of the pairs can be found in stats, then stop
      break

    pair = min(stats, key = lambda p: self.merges.get(p, float("inf"))) # find the pair with the lowest merge index + ignores all unlearned pairs (sets them to infinity so they get ignored by min())
    id = self.merges[pair]
    ids = merge(ids, pair, id)

  return ids

*Now that we have all of our compnents created, we can combine them all into one class `BasicTokenizer`.*

In [ ]:
# HELPER FUNCTIONS

def get_stats(ids):
    stats = {}
    for i in range(len(ids) - 1):
      pair = (ids[i], ids[i + 1])
      stats[pair] = stats.get(pair, 0) + 1

    return stats

def merge(ids, pair, new_id):
  merged_stats = []
  i = 0

  while i < len(ids):
    if i == len(ids) - 1:
      merged_stats.append(ids[i])
      break

    if ids[i] == pair[0] and ids[i + 1] == pair[1]:
      merged_stats.append(new_id)
      i += 2
    else:
      merged_stats.append(ids[i])
      i += 1

  return merged_stats

# MAIN FUNCTION

class BasicTokenizer:

  def train(self, text, vocab_size, verbose = False):
    ids = list(text.encode("utf-8"))

    self.merges = {}
    self.vocab = {}
    for i in range(256):
      self.vocab[i] = bytes([i])

    for i in range(vocab_size - 256):
      stats = get_stats(ids)
      pair = max(stats, key = stats.get)
      ids = merge(ids, pair, 256 + i)

      self.merges[pair] = 256 + i
      self.vocab[256 + i] = self.vocab[pair[0]] + self.vocab[pair[1]]

  def decode(self, ids):
    byte_seq = b"".join(self.vocab[id] for id in ids)
    return byte_seq.decode("utf-8")

  def encode(self, text):
    ids = list(text.encode("utf-8"))

    while(True):
      stats = get_stats(ids)
      if not any(p in self.merges for p in stats):
        break

      pair = min(stats, key = lambda p: self.merges.get(p, float("inf")))
      id = self.merges[pair]
      ids = merge(ids, pair, id)

    return ids

*Now that we have our whole class together, let's try to run it with a small test case.*

In [ ]:
tok = BasicTokenizer()
tok.train("hello world hello", vocab_size = 260)  # creates 4 new merged tokens
ids = tok.encode("hello world")
print(ids)
print(tok.decode(ids))

[259, 32, 119, 111, 114, 108, 100]
hello world


*We can see what the tokenization looks like:*

In [ ]:
for id in [259, 32, 119, 111, 114, 108, 100]:
    print(id, tok.vocab[id])

259 b'hello'
32 b' '
119 b'w'
111 b'o'
114 b'r'
108 b'l'
100 b'd'


*See below for the merging of the word "hello".*

In [ ]:
for pair, id in tok.merges.items():
    print(f"{tok.vocab[pair[0]]} + {tok.vocab[pair[1]]} -> {tok.vocab[id]}")

b'h' + b'e' -> b'he'
b'he' + b'l' -> b'hel'
b'hel' + b'l' -> b'hell'
b'hell' + b'o' -> b'hello'


*And with that, we have completed step 1!*

---
**Step 2**  
Convert your `BasicTokenizer` into `RegexTokenizer`, which takes a regex pattern and splits the text exactly as GPT-4 would. Process the parts separately as before, then concatenate the results.  
Retrain your tokenizer and compare the results before and after. You should see that you will now have no tokens that go across categories (numbers, letters, punctuation, more than one whitespace).  

Use the GPT-4 pattern:
```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```
---

*Install `regex`*

In [ ]:
!pip install regex

*We can test what `regex` does. One of the largest issues our `BasicTokenizer` has is that it will merge ANY character. This is an issue when it comes to apostrophes, prefixes & suffixes, and even punctuation. `regex` essentially splits so that the tokenizer should, ideally, iterate each independent chunk instead of the entire string. The `GPT4_SPLIT_PATTERN` is a collection of what parts we want to break apart. For instance, `\p{N}` matches every single numeric number.*

In [ ]:
import regex

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

text = "Hello world! 123 how're you?"
chunks = regex.findall(GPT4_SPLIT_PATTERN, text)
print(chunks)

['Hello', ' world', '!', ' ', '123', ' how', "'re", ' you', '?']


*What is relatively handy in trying to create `RegexTokenizer` is that it is just like `BasicTokenizer`, except we want to iterate through each chunk individually rather than the entire sequence at once. As such, the helper functions and `decode` will remain the same.*

*First, `train`. We want to use `regex.findall` to seperate the text into appropriate chunks. Then, we want to put it into `ids`, just like we did for `BasicTokenizer`. However, now we note that `ids` is a list of lists, which means our helper functions will not properly work. To handle this issue, we just have to iterate the helper functions over ALL chunks. For every chunk in `ids`, we want to add up the frequency of the pairs (embodied as `count` in the code). This retains the same effect as calculating the most frequent pair (and the pair we want to merge). Lastly, we replace `ids = merge(part, pair, 256 + i)` with `ids = [merge(part, pair, 256 + i) for id in ids]` so it properly merges the pair across all chunks*

*Second, `encode`. It borrows the same code and idea from the previous function. Since we made `ids` a list of lists, we want to make sure our helper functions iteratre through each chunk properly.*

In [ ]:
class RegexTokenizer:
  def train(self, text, vocab_size, verbose = False):

    # get all chunks created by regex and put it into ids
    ids = []
    chunks = regex.findall(GPT4_SPLIT_PATTERN, text)
    for part in chunks:
      ids.append(list(part.encode("utf-8")))

    self.merges = {}
    self.vocab = {}
    for i in range(256):
      self.vocab[i] = bytes([i])

    for i in range(vocab_size - 256):
      # sums up all pair appearances by count (v)
      stats = {}
      for part in ids:
          for pair, count in get_stats(part).items():
              stats[pair] = stats.get(pair, 0) + count

      pair = max(stats, key = stats.get)

      # merges through EACH chunk
      ids = [merge(part, pair, 256 + i) for part in ids]

      self.merges[pair] = 256 + i
      self.vocab[256 + i] = self.vocab[pair[0]] + self.vocab[pair[1]]

  def decode(self, ids):
    byte_seq = b"".join(self.vocab[id] for id in ids)
    return byte_seq.decode("utf-8")

  def encode(self, text):

    # gathers all chunks
    ids = []
    chunks = regex.findall(GPT4_SPLIT_PATTERN, text)
    for part in chunks:
      ids.append(list(part.encode("utf-8")))

    while(True):
      # instead of summing across the entire pair, sum the pair from each chunk
      stats = {}
      for part in ids:
          for pair, count in get_stats(part).items():
              stats[pair] = stats.get(pair, 0) + count
      if not any(p in self.merges for p in stats):
        break

      pair = min(stats, key = lambda p: self.merges.get(p, float("inf")))
      id = self.merges[pair]

      # merge across all chunks independently
      ids = [merge(part, pair, id) for part in ids]

    return [id for part in ids for id in part]  # IMPORTANT: return ALL chunks

*Once more, we can test our code with a test prompt. This time, we incorporate a contraction, numbers, and different punctuation.*

In [ ]:
tok = RegexTokenizer()
tok.train("hello world! 123 how're you?", vocab_size = 270)
ids = tok.encode("hello world!")

print(ids)
print(tok.decode(ids))

[259, 264, 33]
hello world!


In [ ]:
for id in [259, 264, 33]:
    print(id, tok.vocab[id])

259 b'hello'
264 b' world'
33 b'!'


*Step 2 is now complete!*

---
**Step 3**  
You're now ready to load the merges from the GPT-4 tokenizer and show that your tokenizer produces the identical results for both encode and decode, matching [tiktoken](https://github.com/openai/tiktoken).
```
# match this
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) # get the same text back
Unfortunately, you will run into two issues:
```

It is not trivial to recover the raw merges from the GPT-4 tokenizer. You can easily recover what we call vocab here, and what they call and store under 'enc._mergeable_ranks'. Feel free to copy paste the 'recover_merges' function in 'minbpe/gpt4.py', which takes these ranks and returns the raw merges. If you wish to know how this function works, read [this](https://github.com/openai/tiktoken/issues/60) and [this](https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306). Basically, under some conditions it is enough to only store the parent nodes (and their rank) and get rid of the precise details of which children merged up to any parent.  
Second, the GPT-4 tokenizer for some reason permutes its raw bytes. It stores this permutation in the first 256 elements of the mergeable ranks, so you can recover this byte shuffle relatively simply as `byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}`. In both your encode and decode, you'll have to shuffle bytes around accordingly. If you're stuck, reference the `minbpe/gpt4.py` file for hints.

---

*Install tiktoken*

In [ ]:
!pip install tiktoken

*The goal of this step is to produce the exact same token IDs as the example below.*  
*NOTE: `cl100k_base` is the tokenizer used by GPT-4.*

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids)
print(ids)
print(text)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
hello world!!!? (안녕하세요!) lol123 😉


*For time and simplicity's sake, the following code is copied from `minbpe/gpt4.py`.*

In [ ]:
def bpe(mergeable_ranks, token, max_rank):
    # helper function used in get_gpt4_merges() to reconstruct the merge forest
    parts = [bytes([b]) for b in token]
    while True:
        min_idx = None
        min_rank = None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break
        assert min_idx is not None
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]
    return parts

def recover_merges(mergeable_ranks):
    # the `merges` are already the byte sequences in their merged state.
    # so we have to recover the original pairings. We can do this by doing
    # a small BPE training run on all the tokens, in their order.
    # also see https://github.com/openai/tiktoken/issues/60
    # also see https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue # skip raw bytes
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))
        assert len(pair) == 2
        # recover the integer ranks of the pair
        ix0 = mergeable_ranks[pair[0]]
        ix1 = mergeable_ranks[pair[1]]
        merges[(ix0, ix1)] = rank

    return merges

In [ ]:
mergeable_ranks = enc._mergeable_ranks
merges = recover_merges(mergeable_ranks)
print(len(merges), "merges")  # should output 100,000, matching GPT-4's vocabulary of 100,256 (256 raw bytes)

100000 merges


In [ ]:
# GPT-4 shuffles the bytes around. Raw byte `i` should actually be treated as ID `byte_shuffle[i]`.
byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}

*We want to create a `GPT4Tokenizer` class that extends from `RegexTokenizer` and hardcodes the merge and the byte shuffle. We do this so that we can match tiktoken exactly.*

*We first start by initializing our tokenizer via `__init__`. The only fundamental difference from the previous tokenizers is that GPT-4 uses `byte_shuffle`, which shuffles the bytes so that raw bytes are mapped to non-sequential IDs (once more, we only use this to match the exact token format as tiktoken).*  
*To make sure our shuffled IDs can map back to its original format, we just set `self.vocab[byte_shuffle[i]] = bytes([i])`. Now, we can just copy over the `encode` function we built.*

*Consistent with our idea of shuffling the bytes before any merge, we also need to do the same for each chunk during encoding. So, instead of just encoding the chunks with `UTF-8`,  we need to shuffle them first. Code-wise, it looks like `[self.byte_shuffle[b] for b in part.encode("utf-8")]`. And lastly, it is important to add that to `ids`, just like we did for `RegexTokenizer`.*

In [ ]:
class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        self.merges = merges
        self.vocab = {}
        for i in range(256):
            self.vocab[byte_shuffle[i]] = bytes([i])  # shuffled ID maps back to the original byte
        self.byte_shuffle = byte_shuffle

        # builds the vocab for all merged tokens
        # the bytes that token `rank` represent is equal to the bytes of `p0 + p1`
        for (p0, p1), rank in self.merges.items():
          self.vocab[rank] = self.vocab[p0] + self.vocab[p1]

    def encode(self, text):
      ids = []
      chunks = regex.findall(GPT4_SPLIT_PATTERN, text)
      for part in chunks:
        ids.append([self.byte_shuffle[b] for b in part.encode("utf-8")])  # each byte `b` in the chunk needs to be replaced by the byte shuffle function

      while(True):
        stats = {}
        for part in ids:
            for pair, count in get_stats(part).items():
                stats[pair] = stats.get(pair, 0) + count
        if not any(p in self.merges for p in stats):
          break

        pair = min(stats, key = lambda p: self.merges.get(p, float("inf")))
        id = self.merges[pair]

        ids = [merge(part, pair, id) for part in ids]

      return [id for part in ids for id in part]

*Our code is finished. To ensure that we pass, our tokenizer model must return the same tokens as `tiktoken` does. See below.*

In [ ]:
gpt4 = GPT4Tokenizer()

testTXT = "hello world!!!? (안녕하세요!) lol123 😉"

pred = gpt4.encode(testTXT) # our tokenizer IDs
exp = enc.encode(testTXT)   # tiktoken's tokenizer IDs

print(pred)
print(exp)
print(pred == exp)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
True


*Since `pred == exp`, that means our tokenizer works!*

*We have now completed Step 3!*

---
**Step 4**  
(Optional, irritating, not obviously useful) Add the ability to handle special tokens. You'll then be able to match the output of tiktoken even when special tokens are present, e.g.:
```
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
```

Without `allowed_special`, tiktoken will error.

---

*What is a special token?*

*A special token is a string like `<|endoftext|>` that should never be split by BPE. It should always map to a specific ID.*

*First, store these special tokens in a dictionary*  
*Secondly, in the `encode` function, we want to split the text on special tokens first, encode the normal tokens, and then reinsert our special tokens.*

*Refer to the snippet below as a demonstration of how regex should treat special tokens.*

In [ ]:
pattern = "|".join(regex.escape(token) for token in {"<|endoftext|>": 100257})

chunks = regex.split(f"({pattern})", "<|endoftext|>hello world")
print(chunks)

['', '<|endoftext|>', 'hello world']


In [ ]:
class RegexTokenizer:
  # creates a function to register special tokens
  def register_special_tokens(self, special_tokens):
    self.special_tokens = special_tokens
    for token, id in special_tokens.items():
        self.vocab[id] = token.encode("utf-8")

  def train(self, text, vocab_size, verbose = False):

    self.special_tokens = {}  # initialized even if `register_special_tokens` is never called

    ids = []
    chunks = regex.findall(GPT4_SPLIT_PATTERN, text)
    for part in chunks:
      ids.append(list(part.encode("utf-8")))

    self.merges = {}
    self.vocab = {}
    for i in range(256):
      self.vocab[i] = bytes([i])

    for i in range(vocab_size - 256):
      stats = {}
      for part in ids:
          for pair, count in get_stats(part).items():
              stats[pair] = stats.get(pair, 0) + count

      pair = max(stats, key = stats.get)
      ids = [merge(part, pair, 256 + i) for part in ids]

      self.merges[pair] = 256 + i
      self.vocab[256 + i] = self.vocab[pair[0]] + self.vocab[pair[1]]

  def decode(self, ids):
    byte_seq = b"".join(self.vocab[id] for id in ids)
    return byte_seq.decode("utf-8")

  def encode(self, text):
    pattern = "|".join(regex.escape(token) for token in self.special_tokens)  # joins every special token; regex.escape tells regex to treat special characters literally instead of as regex syntax
    chunks = regex.split(f"({pattern})", text)  # by saving our pattern, it prevents regex from splitting on the special token and throwing it out

    ids = []
    for chunk in chunks:
        # we want to add the special token as is to our ids list
        if chunk in self.special_tokens:
            ids.append(self.special_tokens[chunk])
        # if it is just a normal token, then do BPE
        elif chunk:
            chunk_ids = list(chunk.encode("utf-8"))
            # our previous BPE logic
            while True:
                stats = get_stats(chunk_ids)
                if not any(p in self.merges for p in stats):
                    break
                pair = min(stats, key = lambda p: self.merges.get(p, float("inf")))
                chunk_ids = merge(chunk_ids, pair, self.merges[pair])
            ids.extend(chunk_ids)

    return ids

*Let's test it!*

In [ ]:
tok = RegexTokenizer()
tok.train("hello world", vocab_size = 260)
tok.register_special_tokens({"<|endoftext|>": 100257})

ids = tok.encode("<|endoftext|>hello world")
print(ids)

[100257, 259, 32, 119, 111, 114, 108, 100]


In [ ]:
ids = [100257, 259, 32, 119, 111, 114, 108, 100]
for id in ids:
  print(id, tok.vocab[id])

100257 b'<|endoftext|>'
259 b'hello'
32 b' '
119 b'w'
111 b'o'
114 b'r'
108 b'l'
100 b'd'


*We can test it again. This time, let's expand our `vocab_size` to 265 so we can BPE the word "world".*

In [ ]:
tok = RegexTokenizer()
tok.train("hello world", vocab_size = 265)
tok.register_special_tokens({"<|endoftext|>": 100257})

ids = tok.encode("<|endoftext|>hello world")
print(ids)

[100257, 259, 264]


In [ ]:
ids = [100257, 259, 264]
for id in ids:
  print(id, tok.vocab[id])

100257 b'<|endoftext|>'
259 b'hello'
264 b' world'


*With that, we are done Step 4 and, consequently, the exercise. Hoorah!*

By Justin Mo